# 📘 Guide 11: Async Python

**Who it's for:** everyone, before week 3. Grading 30 students × 5 questions one call at a time is slow; most of that time is spent *waiting* for the model. `async` lets us wait for many calls at once. **Time:** about 25 minutes.

### 🎯 You'll learn
1. Why waiting is the bottleneck (I/O-bound work)
2. `async def`, `await` and coroutines
3. Running many tasks at once with `asyncio.gather`
4. Limiting concurrency with a `Semaphore` (rate limits!)
5. Handling errors in concurrent work, and async in notebooks vs scripts

## Part 1: The problem

Imagine Emma's 10 answers each take about 1 second for the model to grade. Done one after another, that's 10 seconds. We'll simulate the model with `time.sleep` / `asyncio.sleep`, so no API key is needed.

In [ ]:
import time

ANSWERS = [f"Answer {i}" for i in range(1, 11)]


def grade_sync(answer: str) -> str:
    time.sleep(0.3)  # pretend the model takes 0.3 s
    return f"{answer}: graded"


start = time.perf_counter()
results = [grade_sync(a) for a in ANSWERS]
print(f"🐢 One at a time: {len(results)} answers in {time.perf_counter() - start:.1f}s")

## Part 2: `async def` and `await`

- `async def` defines a **coroutine function**. Calling it doesn't run it: it returns a coroutine object.
- `await` runs a coroutine, and while it's waiting (e.g. on the network) lets *other* coroutines make progress.
- `asyncio.sleep` is the async version of `time.sleep`: it waits *without blocking* everything else.

In [ ]:
import asyncio
import random


async def grade(answer: str) -> str:
    await asyncio.sleep(0.3 + random.random() * 0.2)  # 💡 the "model call"
    return f"{answer}: graded"


coro = grade("Answer 1")
print("Calling it gives:", type(coro).__name__)
print("Awaiting it gives:", await coro)  # 💡 notebooks allow top-level await

## Part 3: `asyncio.gather`: all at once

In [ ]:
start = time.perf_counter()
results = await asyncio.gather(*(grade(a) for a in ANSWERS))
print(f"🚀 Concurrently: {len(results)} answers in {time.perf_counter() - start:.1f}s")
print(results[:3], "...")  # 💡 results come back in the same order as the inputs

Same 10 answers, about one answer's worth of time. With LangChain, the async version of `llm.invoke(...)` is `await llm.ainvoke(...)`, and `llm.abatch([...])` does the gathering for you:

```python
replies = await asyncio.gather(*(llm.ainvoke(prompt_for(a)) for a in answers))
```

## Part 4: Don't overwhelm the API: `Semaphore`

Providers limit how many requests you can make per minute. Firing 500 requests at once will earn you `429` errors. A semaphore is a bouncer that lets only N coroutines in at a time:

In [ ]:
limit = asyncio.Semaphore(3)
in_flight = 0


async def polite_grade(answer: str) -> str:
    global in_flight
    async with limit:  # 💡 at most 3 inside this block at once
        in_flight += 1
        peak = in_flight
        result = await grade(answer)
        in_flight -= 1
    return f"{result} (in flight: {peak})"


start = time.perf_counter()
for line in await asyncio.gather(*(polite_grade(a) for a in ANSWERS)):
    print(line)
print(f"⏱️ {time.perf_counter() - start:.1f}s with at most 3 at a time")

## Part 5: When one task fails

By default, one exception in `gather` raises straight away. With `return_exceptions=True`, you get errors back as values and can handle them answer by answer, so one unreadable script doesn't sink the whole class:

In [ ]:
async def fragile_grade(answer: str) -> str:
    if answer == "Answer 7":
        await asyncio.sleep(0.1)
        raise ValueError("handwriting unreadable")
    return await grade(answer)


outcomes = await asyncio.gather(*(fragile_grade(a) for a in ANSWERS), return_exceptions=True)
for answer, outcome in zip(ANSWERS, outcomes, strict=True):
    if isinstance(outcome, Exception):
        print(f"⚠️ {answer}: needs Emma's review ({outcome})")
ok = sum(not isinstance(o, Exception) for o in outcomes)
print(f"✅ {ok}/{len(ANSWERS)} graded automatically")

## Part 6: Notebooks vs scripts

Notebooks already run an event loop, so you can write `await` at the top level, as we did above. In a `.py` file there is no loop yet, so you start one with `asyncio.run`:

```python
# grade_all.py  (run with: uv run python grade_all.py)
import asyncio

async def main():
    results = await asyncio.gather(*(grade(a) for a in ANSWERS))
    print(results)

if __name__ == "__main__":
    asyncio.run(main())
```

⚠️ Calling `asyncio.run(...)` *inside* a notebook raises "cannot be called from a running event loop". Use `await` there instead.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Try it</h2>
            <span style="color:#f59e0b;">Change the semaphore to 1, 5 and 10 and compare the timings. Then add a <code>timeout</code>: wrap each call in <code>asyncio.wait_for(grade(a), timeout=0.35)</code> and report which answers timed out.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• asyncio documentation: <a href="https://docs.python.org/3/library/asyncio.html">docs.python.org/3/library/asyncio.html</a><br/>• Real Python: async IO in Python: <a href="https://realpython.com/async-io-python/">realpython.com/async-io-python/</a><br/>• LangChain: async runnables: <a href="https://python.langchain.com/docs/concepts/async/">python.langchain.com/docs/concepts/async/</a></span>
        </td>
    </tr>
</table>